# FIFA World Cup Predictor

## Version 3 - Elo + Recent Form Model

### Objective
Combine long-term team strength (Elo) with short-term team performance (Recent Form).

### Features
- home_elo
- away_elo
- elo_difference
- home_form
- away_form
- neutral_encoded

### What is "Recent Form"?

Recent form measures how well a team has played in its **last 5 matches**. Each match is scored as:
- Win = 1.0
- Draw = 0.5
- Loss = 0.0

Form = the average of the last 5 scores.

Example: W W D L W = (1 + 1 + 0.5 + 0 + 1) / 5 = 0.7

### Previous Results

| Experiment | Accuracy |
|------------|----------|
| Team IDs | 51.88% |
| Team IDs + Neutral | 50.72% |
| Elo Random Forest | 51.29% |
| Simple Elo Rule | 55.42% |

### Hypothesis
Recent form will improve the Random Forest model when combined with Elo ratings.

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# Model Training
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

In [2]:
df = pd.read_csv("../data/raw/results.csv")

print(df.shape)
df.head()

(49477, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49413 non-null  float64
 4   away_score  49413 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [4]:
df.isnull().sum()

date           0
home_team      0
away_team      0
home_score    64
away_score    64
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [5]:
df['tournament'].value_counts().head(20)

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

In [6]:
# Drop the 64 future World Cup fixtures that have no score yet
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)

print(df.shape)

df.isnull().sum()

(49413, 9)


date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [7]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 2      # Home Win
    elif row['home_score'] < row['away_score']:
        return 0      # Away Win
    else:
        return 1      # Draw

In [8]:
df['result'] = df.apply(get_result, axis=1)

df['result'].value_counts()

result
2    24216
0    13961
1    11236
Name: count, dtype: int64

In [9]:
df.neutral.value_counts()

neutral
False    36350
True     13063
Name: count, dtype: int64

In [10]:
df['neutral_encoded'] = df['neutral'].astype(int)
df[['neutral', 'neutral_encoded']].head()

,neutral,neutral_encoded
0,False,0
1,False,0
2,False,0
3,False,0
4,False,0


# Step 1 - Build Elo Ratings (unchanged from Version 2)

Elo ratings capture **long-term team strength**. The code below is exactly the same as Version 2:

- Every team starts at 1500.
- After each match, both teams' ratings are updated (K-factor = 20).
- We save each team's rating **BEFORE** the match, so the model never sees the future.

This is our first defence against **data leakage**.

In [11]:
teams = pd.concat([
    df['home_team'],
    df['away_team']
    ]).unique()

elo_rating = {
    team:1500
    for team in teams
    }

In [12]:
print(elo_rating['Brazil'])
print(elo_rating['Germany'])

1500
1500


In [13]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=20):
    return rating + k * (actual - expected)

In [14]:
home_elos = []
away_elos = []

In [15]:
# Work through the matches in chronological order
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values('date').reset_index(drop=True)

In [16]:
for _, row in df.iterrows():

    home_team = row['home_team']
    away_team = row['away_team']

    home_elo = elo_rating[home_team]
    away_elo = elo_rating[away_team]

    # Save Elo BEFORE the match
    home_elos.append(home_elo)
    away_elos.append(away_elo)

    # Expected results
    expected_home = expected_score(home_elo, away_elo)
    expected_away = expected_score(away_elo, home_elo)

    # Actual result
    if row['result'] == 2:      # Home win
        actual_home = 1
        actual_away = 0

    elif row['result'] == 0:    # Away win
        actual_home = 0
        actual_away = 1

    else:                       # Draw
        actual_home = 0.5
        actual_away = 0.5

    # Update ratings
    elo_rating[home_team] = update_elo(
        home_elo,
        expected_home,
        actual_home
    )

    elo_rating[away_team] = update_elo(
        away_elo,
        expected_away,
        actual_away
    )

In [17]:
print(len(home_elos))
print(len(away_elos))

49413
49413


In [18]:
df['home_elo'] = home_elos
df['away_elo'] = away_elos

In [19]:
df[['home_team', 'away_team', 'home_elo', 'away_elo']].head(10)

,home_team,away_team,home_elo,away_elo
0,Scotland,England,1500.000000,1500.000000
1,England,Scotland,1500.000000,1500.000000
2,Scotland,England,1490.000000,1510.000000
3,England,Scotland,1499.424989,1500.575011
4,Scotland,England,1500.541911,1499.458089
5,Scotland,Wales,1510.510716,1500.000000
6,England,Scotland,1489.489284,1520.208286
7,Wales,Scotland,1490.302430,1529.326419
8,Scotland,England,1538.207918,1480.371151
9,Scotland,Wales,1546.558450,1481.420932


In [20]:
df[['home_elo', 'away_elo']].describe()

,home_elo,away_elo
count,49413.000000,49413.000000
mean,1552.886850,1543.885666
std,142.675951,140.839944
min,1037.318348,1041.997000
25%,1459.686152,1454.958676
50%,1547.889975,1541.067828
75%,1646.662833,1638.440255
max,2017.346980,2021.807600


In [21]:
df['elo_difference'] = df['home_elo'] - df['away_elo']

In [22]:
# Reference baseline from Version 2
# Simple rule: the team with the higher Elo is predicted to win.
df['elo_prediction'] = np.where(
    df['elo_difference'] > 0,
    2,   # Home win
    0    # Away win
)

print("Simple Elo Rule Accuracy:", accuracy_score(df['result'], df['elo_prediction']))

Simple Elo Rule Accuracy: 0.554165907757068


# Step 2 - Build Recent Form Features

Now we add **recent form** - a measure of how a team has performed in its last 5 matches.

This is a separate loop over the (chronological) dataframe. For each match we do three things:

1. Look at each team's history of past results (only matches **BEFORE** the current one) and store the form.
2. Work out the points for the current match (Win = 1.0, Draw = 0.5, Loss = 0.0).
3. Only **after** storing the form do we add the current match's result to each team's history.

Step 3 is the important one: the current match's result is **not** included in its own form.
That is how we prevent data leakage here.

Teams with fewer than 5 previous matches use whatever matches are available (we never invent results).
A team with no previous matches gets form 0.0.

In [23]:
# How many recent matches to look back
FORM_WINDOW = 5

# Each team's recent results, stored as points (oldest first)
team_form_history = {team: [] for team in teams}

home_forms = []
away_forms = []

for _, row in df.iterrows():

    home_team = row['home_team']
    away_team = row['away_team']

    # 1) Form from matches BEFORE this one (no leakage!)
    home_history = team_form_history[home_team]
    away_history = team_form_history[away_team]

    home_form = np.mean(home_history) if len(home_history) > 0 else 0.0
    away_form = np.mean(away_history) if len(away_history) > 0 else 0.0

    # Save form BEFORE updating the histories
    home_forms.append(home_form)
    away_forms.append(away_form)

    # 2) Points for the current match
    if row['result'] == 2:      # Home win
        home_points = 1.0
        away_points = 0.0

    elif row['result'] == 0:    # Away win
        home_points = 0.0
        away_points = 1.0

    else:                       # Draw
        home_points = 0.5
        away_points = 0.5

    # 3) NOW update each team's history with the current result
    home_history.append(home_points)
    away_history.append(away_points)

    # Keep only the last 5 matches
    if len(home_history) > FORM_WINDOW:
        home_history.pop(0)
    if len(away_history) > FORM_WINDOW:
        away_history.pop(0)

In [24]:
df['home_form'] = home_forms
df['away_form'] = away_forms

df[['date', 'home_team', 'away_team', 'home_form', 'away_form', 'result']].head(10)

,date,home_team,away_team,home_form,away_form,result
0,1872-11-30,Scotland,England,0.00,0.00,1
1,1873-03-08,England,Scotland,0.50,0.50,2
2,1874-03-07,Scotland,England,0.25,0.75,2
3,1875-03-06,England,Scotland,0.50,0.50,1
4,1876-03-04,Scotland,England,0.50,0.50,2
5,1876-03-25,Scotland,Wales,0.60,0.00,2
6,1877-03-03,England,Scotland,0.40,0.70,0
7,1877-03-05,Wales,Scotland,0.00,0.90,0
8,1878-03-02,Scotland,England,0.90,0.30,2
9,1878-03-23,Scotland,Wales,1.00,0.00,2


# Step 2b - Sanity Checks for the Form Features

Before trusting the new features, let's verify they behave correctly:

1. The example W W D L W should equal 0.7.
2. Form values should be between 0 and 1, with no missing values.
3. The very first match (no history) should have form 0.0.
4. We manually recompute one match's form from earlier matches only (leakage check).

In [25]:
# Check 1: W W D L W = 0.7
example = np.mean([1.0, 1.0, 0.5, 0.0, 1.0])
print("W W D L W =", example)

W W D L W = 0.7


In [26]:
# Check 2: no missing values, and values stay between 0 and 1
print(df[['home_form', 'away_form']].isnull().sum())

print()

print(df[['home_form', 'away_form']].describe())

home_form    0
away_form    0
dtype: int64

          home_form     away_form
count  49413.000000  49413.000000
mean       0.502382      0.493356
std        0.240271      0.240957
min        0.000000      0.000000
25%        0.300000      0.300000
50%        0.500000      0.500000
75%        0.700000      0.700000
max        1.000000      1.000000


In [27]:
# Check 3: the very first match has no history, so form must be 0.0
df[['date', 'home_team', 'away_team', 'home_form', 'away_form']].head(3)

,date,home_team,away_team,home_form,away_form
0,1872-11-30,Scotland,England,0.00,0.00
1,1873-03-08,England,Scotland,0.50,0.50
2,1874-03-07,Scotland,England,0.25,0.75


In [28]:
# Check 4: no leakage - manually recompute form for one match
# using ONLY matches that happened before it.

# Pick an England home match with enough history
idx = df[df['home_team'] == 'England'].index[100]
match_date = df.loc[idx, 'date']

# England's 5 most recent matches BEFORE this one
prior_matches = df[
    (df['date'] < match_date) &
    ((df['home_team'] == 'England') | (df['away_team'] == 'England'))
].tail(5)

points = []
for _, m in prior_matches.iterrows():
    if m['home_team'] == 'England':
        points.append({0: 0.0, 1: 0.5, 2: 1.0}[m['result']])
    else:
        points.append({0: 1.0, 1: 0.5, 2: 0.0}[m['result']])

manual_form = np.mean(points)

print("Match date:", match_date.date())
print("Prior matches used:", len(prior_matches))
print("Manual form from prior matches only:", manual_form)
print("home_form in the dataframe:        ", df.loc[idx, 'home_form'])
print("Match?", manual_form == df.loc[idx, 'home_form'])

Match date: 1946-04-24
Prior matches used: 5
Manual form from prior matches only: 0.4
home_form in the dataframe:         0.4
Match? True


# Step 3 - Prepare Features and Train/Test Split

Now we combine the Elo features (Version 2) with the new form features.
We use the exact same model settings as Version 2, so that any change in accuracy is caused by the new features - not by tuning.

In [29]:
feature_cols = [
    'home_elo',
    'away_elo',
    'elo_difference',
    'home_form',
    'away_form',
    'neutral_encoded'
]

X = df[feature_cols]
y = df['result']

print(X.shape)
X.head()

(49413, 6)


,home_elo,away_elo,elo_difference,home_form,away_form,neutral_encoded
0,1500.000000,1500.000000,0.000000,0.00,0.00,0
1,1500.000000,1500.000000,0.000000,0.50,0.50,0
2,1490.000000,1510.000000,-20.000000,0.25,0.75,0
3,1499.424989,1500.575011,-1.150023,0.50,0.50,0
4,1500.541911,1499.458089,1.083822,0.50,0.50,0


In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(39530, 6)
(9883, 6)


# Step 4 - Train the Random Forest

Same model as Version 2: Random Forest with 200 trees and random_state=42. No hyperparameter tuning - we only changed the features.

In [31]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

# Step 5 - Evaluate the Model

We measure accuracy and print a classification report (same as the previous versions).

In [32]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

Accuracy: 0.541030051603764


In [33]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.49      0.51      0.50      2734
           1       0.26      0.11      0.15      2324
           2       0.61      0.77      0.68      4825

    accuracy                           0.54      9883
   macro avg       0.45      0.46      0.44      9883
weighted avg       0.49      0.54      0.51      9883



In [34]:
# Which features helped the most?
for feature, importance in zip(feature_cols, model.feature_importances_):
    print(f"{feature}: {importance:.4f}")

home_elo: 0.2457
away_elo: 0.2484
elo_difference: 0.3049
home_form: 0.0953
away_form: 0.0927
neutral_encoded: 0.0129


# Summary - Did Recent Form Help?

| Experiment | Accuracy |
|------------|----------|
| Team IDs | 51.88% |
| Team IDs + Neutral | 50.72% |
| Elo Random Forest | 51.29% |
| **Elo + Form Random Forest** | **54.10%** |
| Simple Elo Rule | 55.42% |

What to look at:
- Did the Random Forest accuracy improve compared to 51.29% (Elo only)?
- In the classification report, how does the model do on home wins (class 2) vs draws (class 1)?
- Which features have the highest importance? Do the form features matter?

Note: the simple Elo rule (55.42%) is a strong baseline because it only ever predicts a win for one side - it never tries to predict a draw.